# 00 — Build NIR UCO database

This notebook builds the NIR UCO image-level and object-level databases from the raw `.mat` file.

Main outputs:

- `image_db`: image-level database with cubes, segmentation masks, labels and metadata.
- `object_db`: object-level database with object masks, spectra, geometry and metadata.
- HDF5 database file for reuse in the next notebooks.
- CSV summaries for quick quality control.

In [1]:
from __future__ import annotations

import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook display
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

# If the notebook is launched from notebooks/, project root is parent.
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts
SRC_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src


In [2]:
from src.io.dataload import load_mat_file
from src.data.database import (
    parse_image_key,
    preprocess_nir_uco_cube,
    build_minimal_nir_uco_object_database,
    detect_known_image_keys, 
    resolve_selected_keys,
)
from src.io.database_h5 import (
    save_nir_uco_h5,
    load_nir_uco_h5,
)
from src.utils import make_wavelengths, save_parquet

from src.visualization.plot_images import (
    plot_label_overlay_from_image_db,
)

from src.visualization.plot_objects import (
    plot_object_grid,
    plot_object_view,
)

from src.visualization.tables import (
    build_database_inventory_table,
)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# ---------------------------------------------------------------------
# Input path
# ---------------------------------------------------------------------
RAW_MAT_PATH = PROJECT_ROOT / "HSI Data" / "NIR camera UCO (889-1702 nm)" / "NIR_uco_sb.mat"

# ---------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------
PROCESSED_DIR = PROJECT_ROOT / "HSI Data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "00_database"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DB_H5_PATH = PROCESSED_DIR / "nir_uco_database.h5"

IMAGE_SUMMARY_PATH = RESULTS_DIR / "image_summary.parquet"
OBJECT_SUMMARY_PATH = RESULTS_DIR / "object_summary.parquet"
DATABASE_MANIFEST_PATH = RESULTS_DIR / "database_manifest.parquet"

# ---------------------------------------------------------------------
# Database construction parameters
# ---------------------------------------------------------------------
WAVELENGTH_MODE = "non_noisy_all"
RESULTS_TAG = "non_noisy_all"

N_START = 889
N_END = 1702
N_BANDS_RAW = 69
N_REMOVE_START = 6
N_STOP_END = None
DATA_MODE = "reflectance"

# Object extraction filter after segmentation.
OBJECT_MIN_AREA = 10

FORCED_SPLIT = "projection"

# Whether to skip images whose name does not match the naming convention.
SKIP_UNKNOWN = True

# If None, all valid 3D arrays from the raw .mat file are processed.
SELECTED_KEYS = None

# HDF5 options
INCLUDE_HEAVY_OBJECT_ARRAYS = False
OVERWRITE_OUTPUTS = True

# ---------------------------------------------------------------------
# Segmentation parameters
# ---------------------------------------------------------------------
SEGMENTATION_KWARGS = {
    "reference_method": "max",
    "threshold_method": "fixed",
    "tau_min": 0.02,
    "min_area": 10,
    "opening_radius": 0,
    "closing_radius": 1,
    "fill_holes": True,
    "min_distance": 10,
    "min_area": OBJECT_MIN_AREA,
    "use_watershed": False,
}

# ---------------------------------------------------------------------
# QC plots
# ---------------------------------------------------------------------
RUN_QC_PLOTS = True
N_QC_IMAGES = 3
N_QC_OBJECTS = 20

print("RAW_MAT_PATH:", RAW_MAT_PATH)
print("DB_H5_PATH:", DB_H5_PATH)
print("IMAGE_SUMMARY_PATH:", IMAGE_SUMMARY_PATH)
print("OBJECT_SUMMARY_PATH:", OBJECT_SUMMARY_PATH)
print("DATABASE_MANIFEST_PATH:", DATABASE_MANIFEST_PATH)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("OBJECT_MIN_AREA:", OBJECT_MIN_AREA)
print("FORCED_SPLIT:", FORCED_SPLIT)
print("SEGMENTATION_KWARGS:", SEGMENTATION_KWARGS)

RAW_MAT_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\NIR camera UCO (889-1702 nm)\NIR_uco_sb.mat
DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
IMAGE_SUMMARY_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\image_summary.parquet
OBJECT_SUMMARY_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\object_summary.parquet
DATABASE_MANIFEST_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\database_manifest.parquet
WAVELENGTH_MODE: non_noisy_all
OBJECT_MIN_AREA: 10
FORCED_SPLIT: projection
SEGMENTATION_KWARGS: {'reference_method': 'max', 'threshold_method': 'fixed', 'tau_min': 0.02, 'min_area': 10, 'opening_radius': 0, 'closing_radius': 1, 'fill_holes': True, 'min_distance': 10, 'use_watershed': False}


In [5]:
if not RAW_MAT_PATH.exists():
    raise FileNotFoundError(
        f"Raw .mat file not found: {RAW_MAT_PATH}\n"
        "Update RAW_MAT_PATH in the parameters cell."
    )

output_paths = [
    DB_H5_PATH,
    IMAGE_SUMMARY_PATH,
    OBJECT_SUMMARY_PATH,
    DATABASE_MANIFEST_PATH,
]

existing_outputs = [p for p in output_paths if p.exists()]

if existing_outputs and not OVERWRITE_OUTPUTS:
    raise FileExistsError(
        "Some output files already exist:\n"
        + "\n".join(str(p) for p in existing_outputs)
        + "\nSet OVERWRITE_OUTPUTS=True if you want to overwrite them."
    )

print("Raw file found.")

Raw file found.


In [6]:
t0 = time.time()

raw_data = load_mat_file(RAW_MAT_PATH)

elapsed = time.time() - t0

print(f"Loaded raw data in {elapsed:.2f} s")
print(f"Number of entries in raw .mat file: {len(raw_data)}")
print("First keys:")
print(list(raw_data.keys())[:20])

Loaded raw data in 18.04 s
Number of entries in raw .mat file: 48
First keys:
['alm1pea1_sb', 'alm1pea2_sb', 'alm1pea3_sb', 'alm1pea4_sb', 'alm2pea1_sb', 'alm2pea2_sb', 'alm2pea3_sb', 'alm2pea4_sb', 'alm3pea1_sb', 'alm3pea2_sb', 'alm3pea3_sb', 'alm3pea4_sb', 'alm4pea1_sb', 'alm4pea2_sb', 'alm4pea3_sb', 'alm4pea4_sb', 'alm5pea1_sb', 'alm5pea2_sb', 'alm5pea3_sb', 'alm5pea4_sb']


In [7]:
raw_entries = []

for key, value in raw_data.items():
    arr = np.asarray(value)
    raw_entries.append({
        "key": key,
        "type": type(value).__name__,
        "ndim": arr.ndim,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "is_candidate_cube": bool(arr.ndim == 3),
    })

raw_entries_df = pd.DataFrame(raw_entries).sort_values(
    ["is_candidate_cube", "key"],
    ascending=[False, True],
).reset_index(drop=True)

raw_entries_df

,key,type,ndim,shape,dtype,is_candidate_cube
0,alm1pea1_sb,ndarray,3,"(370, 318, 69)",float64,True
1,alm1pea2_sb,ndarray,3,"(370, 318, 69)",float64,True
2,alm1pea3_sb,ndarray,3,"(370, 318, 69)",float64,True
3,alm1pea4_sb,ndarray,3,"(370, 318, 69)",float64,True
4,alm2pea1_sb,ndarray,3,"(370, 318, 69)",float64,True
5,alm2pea2_sb,ndarray,3,"(370, 318, 69)",float64,True
6,alm2pea3_sb,ndarray,3,"(370, 318, 69)",float64,True
7,alm2pea4_sb,ndarray,3,"(370, 318, 69)",float64,True
8,alm3pea1_sb,ndarray,3,"(370, 318, 69)",float64,True
9,alm3pea2_sb,ndarray,3,"(370, 318, 69)",float64,True


In [8]:
candidate_keys = raw_entries_df.loc[
    raw_entries_df["is_candidate_cube"],
    "key",
].tolist()

parsed_rows = []

for key in candidate_keys:
    meta = parse_image_key(key)
    arr = np.asarray(raw_data[key])
    parsed_rows.append({
        "original_key": meta["original_key"],
        "clean_key": meta["clean_key"],
        "sample_kind": meta["sample_kind"],
        "nut_type": meta["nut_type"],
        "batch": meta["batch"],
        "position_set": meta["position_set"],
        "is_pure": meta["is_pure"],
        "is_mixture": meta["is_mixture"],
        "is_position_reference": meta["is_position_reference"],
        "is_unknown": meta["is_unknown"],
        "description": meta["description"],
        "shape": arr.shape,
        "n_rows": arr.shape[0],
        "n_cols": arr.shape[1],
        "n_bands_raw": arr.shape[2],
    })

parsed_df = pd.DataFrame(parsed_rows).sort_values(
    ["sample_kind", "nut_type", "batch", "clean_key"],
    na_position="last",
).reset_index(drop=True)

parsed_df

,original_key,clean_key,sample_kind,nut_type,batch,position_set,is_pure,is_mixture,is_position_reference,is_unknown,description,shape,n_rows,n_cols,n_bands_raw
0,alm1pea1_sb,alm1pea1,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 1 + peanut batch 1,"(370, 318, 69)",370,318,69
1,alm1pea2_sb,alm1pea2,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 1 + peanut batch 2,"(370, 318, 69)",370,318,69
2,alm1pea3_sb,alm1pea3,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 1 + peanut batch 3,"(370, 318, 69)",370,318,69
3,alm1pea4_sb,alm1pea4,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 1 + peanut batch 4,"(370, 318, 69)",370,318,69
4,alm2pea1_sb,alm2pea1,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 2 + peanut batch 1,"(370, 318, 69)",370,318,69
5,alm2pea2_sb,alm2pea2,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 2 + peanut batch 2,"(370, 318, 69)",370,318,69
6,alm2pea3_sb,alm2pea3,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 2 + peanut batch 3,"(370, 318, 69)",370,318,69
7,alm2pea4_sb,alm2pea4,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 2 + peanut batch 4,"(370, 318, 69)",370,318,69
8,alm3pea1_sb,alm3pea1,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 3 + peanut batch 1,"(370, 318, 69)",370,318,69
9,alm3pea2_sb,alm3pea2,mixture,mixture,NaN,NaN,False,True,False,False,mixture: almond batch 3 + peanut batch 2,"(370, 318, 69)",370,318,69


In [9]:
if SELECTED_KEYS:
    selected_keys = resolve_selected_keys(raw_data, SELECTED_KEYS)
    detected_rows = [(key, parse_image_key(key)) for key in selected_keys]
    print(f"Selected images: {len(selected_keys)} user-selected image(s)")
else:
    detected_rows = detect_known_image_keys(raw_data, skip_non_cubes=True)
    selected_keys = [key for key, _ in detected_rows]
    print("Selected images: all automatically recognized images")

if not selected_keys:
    raise RuntimeError(
        "No recognized NIR UCO image was found. Check image names and parsing patterns."
    )

detected_df = pd.DataFrame([
    {
        "original_key": key,
        "clean_key": meta["clean_key"],
        "sample_kind": meta["sample_kind"],
        "nut_type": meta["nut_type"],
        "batch": meta["batch"],
        "position_set": meta["position_set"],
        "description": meta["description"],
    }
    for key, meta in detected_rows
])

display(detected_df)

print(f"Number of selected images: {len(selected_keys)}")
print(selected_keys)

Selected images: all automatically recognized images


,original_key,clean_key,sample_kind,nut_type,batch,position_set,description
0,alm1pea1_sb,alm1pea1,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 1
1,alm1pea2_sb,alm1pea2,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 2
2,alm1pea3_sb,alm1pea3,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 3
3,alm1pea4_sb,alm1pea4,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 4
4,alm2pea1_sb,alm2pea1,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 1
5,alm2pea2_sb,alm2pea2,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 2
6,alm2pea3_sb,alm2pea3,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 3
7,alm2pea4_sb,alm2pea4,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 4
8,alm3pea1_sb,alm3pea1,mixture,mixture,NaN,NaN,mixture: almond batch 3 + peanut batch 1
9,alm3pea2_sb,alm3pea2,mixture,mixture,NaN,NaN,mixture: almond batch 3 + peanut batch 2


Number of selected images: 48
['alm1pea1_sb', 'alm1pea2_sb', 'alm1pea3_sb', 'alm1pea4_sb', 'alm2pea1_sb', 'alm2pea2_sb', 'alm2pea3_sb', 'alm2pea4_sb', 'alm3pea1_sb', 'alm3pea2_sb', 'alm3pea3_sb', 'alm3pea4_sb', 'alm4pea1_sb', 'alm4pea2_sb', 'alm4pea3_sb', 'alm4pea4_sb', 'alm5pea1_sb', 'alm5pea2_sb', 'alm5pea3_sb', 'alm5pea4_sb', 'almond1_sb', 'almond2_sb', 'almond3_sb', 'almond4_sb', 'pea1_pos1_sb', 'pea1_pos2_sb', 'pea1_pos3_sb', 'pea1_pos4_sb', 'pea1_pos5_sb', 'pea2_pos1_sb', 'pea2_pos2_sb', 'pea2_pos3_sb', 'pea2_pos4_sb', 'pea2_pos5_sb', 'pea3_pos1_sb', 'pea3_pos2_sb', 'pea3_pos3_sb', 'pea3_pos4_sb', 'pea3_pos5_sb', 'pea4_pos1_sb', 'pea4_pos2_sb', 'pea4_pos3_sb', 'pea4_pos4_sb', 'pea4_pos5_sb', 'peanut1_sb', 'peanut2_sb', 'peanut3_sb', 'peanut4_sb']


In [10]:
image_type_summary = (
    parsed_df
    .groupby(["sample_kind", "nut_type"], dropna=False)
    .size()
    .reset_index(name="n_images")
    .sort_values(["sample_kind", "nut_type"])
)

batch_summary = (
    parsed_df
    .groupby(["sample_kind", "nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_images")
    .sort_values(["sample_kind", "nut_type", "batch"])
)

display(image_type_summary)
display(batch_summary)

,sample_kind,nut_type,n_images
0,mixture,mixture,20
1,position_reference,peanut,20
2,pure,almond,4
3,pure,peanut,4


,sample_kind,nut_type,batch,n_images
0,mixture,mixture,NaN,20
1,position_reference,peanut,1.0,5
2,position_reference,peanut,2.0,5
3,position_reference,peanut,3.0,5
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,1
6,pure,almond,2.0,1
7,pure,almond,3.0,1
8,pure,almond,4.0,1
9,pure,peanut,1.0,1


In [11]:
wavelengths = make_wavelengths(start_nm=N_START, end_nm=N_END, original_bands=N_BANDS_RAW, n_remove_start=N_REMOVE_START, n_stop_end=N_STOP_END)

print("Wavelength axis built from fixed NIR UCO configuration.")
print("START_NM:", N_START)
print("END_NM:", N_END)
print("ORIGINAL_BANDS:", N_BANDS_RAW)
print("N_REMOVE_START:", N_REMOVE_START)
print("N_STOP_END:", N_STOP_END)
print("Number of wavelengths kept:", len(wavelengths))
print("First wavelengths:", wavelengths[:5])
print("Last wavelengths:", wavelengths[-5:])

Wavelength axis built from fixed NIR UCO configuration.
START_NM: 889
END_NM: 1702
ORIGINAL_BANDS: 69
N_REMOVE_START: 6
N_STOP_END: None
Number of wavelengths kept: 63
First wavelengths: [ 960.73529412  972.69117647  984.64705882  996.60294118 1008.55882353]
Last wavelengths: [1654.17647059 1666.13235294 1678.08823529 1690.04411765 1702.        ]


In [12]:
band_check_rows = []

for key in selected_keys:
    raw_cube = np.asarray(raw_data[key])
    clean_cube = preprocess_nir_uco_cube(raw_cube, n_remove_start=N_REMOVE_START, n_stop_end=N_STOP_END)

    band_check_rows.append({
        "key": key,
        "raw_shape": raw_cube.shape,
        "clean_shape": clean_cube.shape,
        "n_removed_start": N_REMOVE_START,
        "n_stop_end": N_STOP_END,
        "raw_min": float(np.nanmin(raw_cube)),
        "raw_max": float(np.nanmax(raw_cube)),
        "clean_min": float(np.nanmin(clean_cube)),
        "clean_max": float(np.nanmax(clean_cube)),
    })

band_check_df = pd.DataFrame(band_check_rows)

display(band_check_df)

if wavelengths is not None:
    expected_n_bands = band_check_df["clean_shape"].iloc[0][2]
    if len(wavelengths) != expected_n_bands:
        raise ValueError(
            f"wavelengths length ({len(wavelengths)}) does not match "
            f"clean cube band count ({expected_n_bands})."
        )

,key,raw_shape,clean_shape,n_removed_start,n_stop_end,raw_min,raw_max,clean_min,clean_max
0,alm1pea1_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.639295,1.254187,-0.059712,1.254187
1,alm1pea2_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.656057,0.983311,-0.022599,0.838548
2,alm1pea3_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.772735,1.065624,-0.026507,0.865334
3,alm1pea4_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.602139,1.308940,-0.022299,1.014218
4,alm2pea1_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.666472,0.991487,-0.032010,0.829993
5,alm2pea2_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-1.063412,1.129765,-0.038481,0.901163
6,alm2pea3_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.758614,0.986584,-0.045499,0.812725
7,alm2pea4_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.508938,1.294200,-0.033658,1.111447
8,alm3pea1_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.601609,1.146167,-0.018868,1.026362
9,alm3pea2_sb,"(370, 318, 69)","(370, 318, 63)",6,None,-0.553906,1.076803,-0.003675,1.076803


In [13]:
print("Building NIR UCO databases...")
print(f"Number of images to process: {len(selected_keys)}")
print("Segmentation parameters:")
print(json.dumps(SEGMENTATION_KWARGS, indent=2))
print("Object extraction min_area:", OBJECT_MIN_AREA)
print("Forced split:", FORCED_SPLIT)

t0 = time.time()

object_db, image_db = build_minimal_nir_uco_object_database(
    data=raw_data,
    selected_keys=selected_keys,
    preprocess_func=preprocess_nir_uco_cube,
    n_remove_start=N_REMOVE_START,
    n_stop_end=N_STOP_END,
    wavelengths=wavelengths,
    data_mode=DATA_MODE,
    min_area=OBJECT_MIN_AREA,
    split=FORCED_SPLIT,
    skip_unknown=SKIP_UNKNOWN,
    segmentation_kwargs=SEGMENTATION_KWARGS,
)

elapsed = time.time() - t0

print("\nDone.")
print(f"Elapsed time: {elapsed:.2f} s")
print(f"Number of images in image_db: {len(image_db)}")
print(f"Number of objects in object_db: {len(object_db)}")

Building NIR UCO databases...
Number of images to process: 48
Segmentation parameters:
{
  "reference_method": "max",
  "threshold_method": "fixed",
  "tau_min": 0.02,
  "min_area": 10,
  "opening_radius": 0,
  "closing_radius": 1,
  "fill_holes": true,
  "min_distance": 10,
  "use_watershed": false
}
Object extraction min_area: 10
Forced split: projection
Processing alm1pea1_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 1, 'token': 'pea'}}


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\data\segmentation.py:118: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=min_area)


  -> 42 objects detected
Processing alm1pea2_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 2, 'token': 'pea'}}
  -> 40 objects detected
Processing alm1pea3_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 3, 'token': 'pea'}}
  -> 40 objects detected
Processing alm1pea4_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 4, 'token': 'pea'}}
  -> 27 objects detected
Processing alm2pea1_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 1, 'token': 'pea'}}
  -> 40 objects detected
Processing alm2pea2_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 2, 'token': 'pea'}}
  -> 40 objects detected
Processing alm2pea3_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 3, 'token': 'pea'}}
  -> 44 objects detected
Processing alm2pea4_sb | kind=mix

In [14]:
database_inventory_df = build_database_inventory_table(
    image_db=image_db,
    object_db=object_db,
)

display(database_inventory_df)

save_parquet(
    database_inventory_df,
    RESULTS_DIR / "database_inventory.parquet",
)

,nut_type,batch,sample_kind,n_objects,n_object_pixels,median_object_area,mean_object_area,n_source_images,n_images
0,mixture,NaN,mixture,NaN,NaN,NaN,NaN,NaN,20.0
1,unknown,NaN,mixture,722.0,63772.0,82.0,88.326870,20.0,NaN
2,peanut,1.0,position_reference,47.0,3388.0,69.0,72.085106,5.0,5.0
3,peanut,2.0,position_reference,47.0,3145.0,65.0,66.914894,5.0,5.0
4,peanut,3.0,position_reference,47.0,2998.0,59.0,63.787234,5.0,5.0
5,peanut,4.0,position_reference,5.0,683.0,127.0,136.600000,5.0,5.0
6,almond,1.0,pure,52.0,4176.0,78.0,80.307692,1.0,1.0
7,almond,2.0,pure,59.0,3753.0,61.0,63.610169,1.0,1.0
8,almond,3.0,pure,55.0,3615.0,63.0,65.727273,1.0,1.0
9,almond,4.0,pure,48.0,5237.0,111.0,109.104167,1.0,1.0


WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/hsi_nuts/results/00_database/database_inventory.parquet')

In [15]:
print("=" * 80)
print("Compatibility check with old script")
print("=" * 80)

print("Expected old-script-compatible settings:")
print(" - N_REMOVE_START :", N_REMOVE_START)
print(" - N_STOP_END     :", N_STOP_END)
print(" - OBJECT_MIN_AREA:", OBJECT_MIN_AREA)
print(" - FORCED_SPLIT   :", FORCED_SPLIT)
print(" - DATA_MODE      :", DATA_MODE)
print(" - wavelengths len:", len(wavelengths))
print()

print("Object split distribution:")
display(
    pd.Series(
        [obj.get("split") for obj in object_db.values()],
        name="split",
    ).value_counts(dropna=False).reset_index(name="n_objects")
)

print("Object area min / median / max:")
areas = np.asarray([obj["area_pixels"] for obj in object_db.values()])
print("min   :", areas.min())
print("median:", np.median(areas))
print("max   :", areas.max())

if areas.min() < OBJECT_MIN_AREA:
    print("[WARNING] Some objects have area below OBJECT_MIN_AREA.")
else:
    print("[OK] No extracted object below OBJECT_MIN_AREA.")

Compatibility check with old script
Expected old-script-compatible settings:
 - N_REMOVE_START : 6
 - N_STOP_END     : None
 - OBJECT_MIN_AREA: 10
 - FORCED_SPLIT   : projection
 - DATA_MODE      : reflectance
 - wavelengths len: 63

Object split distribution:


,split,n_objects
0,projection,1262


Object area min / median / max:
min   : 12
median: 77.0
max   : 224
[OK] No extracted object below OBJECT_MIN_AREA.


In [16]:
image_rows = []

for image_key, img in image_db.items():
    cube = np.asarray(img["cube"])
    labels = np.asarray(img["labels"])
    n_labels = int(labels.max()) if labels.size else 0

    image_rows.append({
        "clean_key": image_key,
        "image_id": img.get("image_id"),
        "sample_kind": img.get("sample_kind"),
        "nut_type": img.get("nut_type"),
        "batch": img.get("batch"),
        "position_set": img.get("position_set"),
        "is_pure": img.get("is_pure"),
        "is_mixture": img.get("is_mixture"),
        "is_position_reference": img.get("is_position_reference"),
        "n_objects": img.get("n_objects"),
        "n_labels": n_labels,
        "height": cube.shape[0],
        "width": cube.shape[1],
        "n_bands": cube.shape[2],
        "threshold": img.get("threshold"),
        "data_mode": img.get("data_mode"),
        "description": img.get("description"),
    })

image_summary_df = (
    pd.DataFrame(image_rows)
    .sort_values(["sample_kind", "nut_type", "batch", "clean_key"], na_position="last")
    .reset_index(drop=True)
)

image_summary_df

,clean_key,image_id,sample_kind,nut_type,batch,position_set,is_pure,is_mixture,is_position_reference,n_objects,n_labels,height,width,n_bands,threshold,data_mode,description
0,alm1pea1,alm1pea1_sb,mixture,mixture,NaN,NaN,False,True,False,42,42,370,318,63,0.02,reflectance,mixture: almond batch 1 + peanut batch 1
1,alm1pea2,alm1pea2_sb,mixture,mixture,NaN,NaN,False,True,False,40,40,370,318,63,0.02,reflectance,mixture: almond batch 1 + peanut batch 2
2,alm1pea3,alm1pea3_sb,mixture,mixture,NaN,NaN,False,True,False,40,40,370,318,63,0.02,reflectance,mixture: almond batch 1 + peanut batch 3
3,alm1pea4,alm1pea4_sb,mixture,mixture,NaN,NaN,False,True,False,27,27,370,318,63,0.02,reflectance,mixture: almond batch 1 + peanut batch 4
4,alm2pea1,alm2pea1_sb,mixture,mixture,NaN,NaN,False,True,False,40,40,370,318,63,0.02,reflectance,mixture: almond batch 2 + peanut batch 1
5,alm2pea2,alm2pea2_sb,mixture,mixture,NaN,NaN,False,True,False,40,40,370,318,63,0.02,reflectance,mixture: almond batch 2 + peanut batch 2
6,alm2pea3,alm2pea3_sb,mixture,mixture,NaN,NaN,False,True,False,44,44,370,318,63,0.02,reflectance,mixture: almond batch 2 + peanut batch 3
7,alm2pea4,alm2pea4_sb,mixture,mixture,NaN,NaN,False,True,False,33,33,370,318,63,0.02,reflectance,mixture: almond batch 2 + peanut batch 4
8,alm3pea1,alm3pea1_sb,mixture,mixture,NaN,NaN,False,True,False,42,42,370,318,63,0.02,reflectance,mixture: almond batch 3 + peanut batch 1
9,alm3pea2,alm3pea2_sb,mixture,mixture,NaN,NaN,False,True,False,39,39,370,318,63,0.02,reflectance,mixture: almond batch 3 + peanut batch 2


In [17]:
object_rows = []

for object_id, obj in object_db.items():
    centroid = obj.get("centroid", (np.nan, np.nan))

    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "image_nut_type": obj.get("image_nut_type"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "position_set": obj.get("position_set"),
        "split": obj.get("split"),
        "is_pure": obj.get("is_pure"),
        "is_mixture": obj.get("is_mixture"),
        "is_position_reference": obj.get("is_position_reference"),
        "label_id": obj.get("label_id"),
        "object_index": obj.get("object_index"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
        "centroid_row": centroid[0] if centroid is not None else np.nan,
        "centroid_col": centroid[1] if centroid is not None else np.nan,
        "bbox": obj.get("bbox"),
        "data_mode": obj.get("data_mode"),
    })

object_summary_df = (
    pd.DataFrame(object_rows)
    .sort_values(["sample_kind", "object_nut_type", "batch", "source_clean_key", "object_index"], na_position="last")
    .reset_index(drop=True)
)

object_summary_df

,object_id,source_clean_key,source_image,sample_kind,image_nut_type,object_nut_type,batch,position_set,split,is_pure,is_mixture,is_position_reference,label_id,object_index,area_pixels,n_pixels,n_bands,centroid_row,centroid_col,bbox,data_mode
0,alm1pea1_obj001,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,1,1,84,84,63,85.726190,45.107143,"(81, 39, 92, 51)",reflectance
1,alm1pea1_obj002,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,2,2,73,73,63,87.739726,123.767123,"(82, 120, 94, 128)",reflectance
2,alm1pea1_obj003,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,3,3,91,91,63,90.032967,157.340659,"(85, 152, 96, 164)",reflectance
3,alm1pea1_obj004,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,4,4,73,73,63,93.589041,90.164384,"(88, 87, 100, 96)",reflectance
4,alm1pea1_obj005,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,5,5,126,126,63,98.825397,70.579365,"(91, 65, 107, 76)",reflectance
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1257,peanut4_obj025,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,25,25,195,195,63,286.112821,96.984615,"(278, 89, 295, 106)",reflectance
1258,peanut4_obj026,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,26,26,99,99,63,294.868687,227.858586,"(287, 223, 302, 234)",reflectance
1259,peanut4_obj027,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,27,27,124,124,63,295.508065,141.338710,"(288, 137, 304, 147)",reflectance
1260,peanut4_obj028,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,28,28,88,88,63,303.704545,52.738636,"(299, 47, 311, 58)",reflectance


In [18]:
print("Image summary by sample kind and nut type")
display(
    image_summary_df
    .groupby(["sample_kind", "nut_type"], dropna=False)
    .agg(
        n_images=("clean_key", "count"),
        n_objects=("n_objects", "sum"),
    )
    .reset_index()
)

print("Object summary by sample kind and object nut type")
display(
    object_summary_df
    .groupby(["sample_kind", "object_nut_type"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        median_area=("area_pixels", "median"),
        min_area=("area_pixels", "min"),
        max_area=("area_pixels", "max"),
    )
    .reset_index()
)

print("Object summary by batch")
display(
    object_summary_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        median_area=("area_pixels", "median"),
    )
    .reset_index()
)

Image summary by sample kind and nut type


,sample_kind,nut_type,n_images,n_objects
0,mixture,mixture,20,722
1,position_reference,peanut,20,146
2,pure,almond,4,214
3,pure,peanut,4,180


Object summary by sample kind and object nut type


,sample_kind,object_nut_type,n_objects,median_area,min_area,max_area
0,mixture,unknown,722,82.0,16,224
1,position_reference,peanut,146,64.5,12,221
2,pure,almond,214,76.0,21,188
3,pure,peanut,180,70.0,15,195


Object summary by batch


,sample_kind,object_nut_type,batch,n_objects,median_area
0,mixture,unknown,NaN,722,82.0
1,position_reference,peanut,1.0,47,69.0
2,position_reference,peanut,2.0,47,65.0
3,position_reference,peanut,3.0,47,59.0
4,position_reference,peanut,4.0,5,127.0
5,pure,almond,1.0,52,78.0
6,pure,almond,2.0,59,61.0
7,pure,almond,3.0,55,63.0
8,pure,almond,4.0,48,111.0
9,pure,peanut,1.0,46,67.5


In [19]:
save_parquet(image_summary_df, IMAGE_SUMMARY_PATH)
save_parquet(object_summary_df, OBJECT_SUMMARY_PATH)

print("Saved essential database summaries:")
print(" -", IMAGE_SUMMARY_PATH)
print(" -", OBJECT_SUMMARY_PATH)

Saved essential database summaries:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\image_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\object_summary.parquet


In [20]:
if RUN_QC_PLOTS:
    qc_image_keys = image_summary_df["clean_key"].head(N_QC_IMAGES).tolist()

    for image_key in qc_image_keys:
        print("QC image:", image_key)
        plot_label_overlay_from_image_db(
            image_db=image_db,
            image_key=image_key,
            base="image_ref",
            title=f"Segmentation labels — {image_key}",
            crop_to_objects=True,
            padding=5,
            show=True,
        )
else:
    print("QC plots skipped.")

QC image: alm1pea1


QC image: alm1pea2


QC image: alm1pea3


In [21]:
if RUN_QC_PLOTS and len(object_db) > 0:
    first_image_key = image_summary_df.loc[
        image_summary_df["n_objects"] > 0,
        "clean_key",
    ].iloc[0]

    print("Object grid for:", first_image_key)

    plot_object_grid(
        object_db,
        source_image=first_image_key,
        title=f"Detected objects — {first_image_key}",
        max_objects=N_QC_OBJECTS,
        show=True,
    )

    first_object_id = object_summary_df.loc[
        object_summary_df["source_clean_key"].eq(first_image_key),
        "object_id",
    ].iloc[0]

    print("Single object view:", first_object_id)

    plot_object_view(
        object_db,
        object_id=first_object_id,
        spectrum_field="mean_spectrum",
        show=True,
    )
else:
    print("Object QC plots skipped.")

Object grid for: alm1pea1


Single object view: alm1pea1_obj001


In [22]:
if OVERWRITE_OUTPUTS and DB_H5_PATH.exists():
    DB_H5_PATH.unlink()

saved_paths = []

saved_h5_path = save_nir_uco_h5(
    object_database=object_db,
    image_database=image_db,
    path=DB_H5_PATH,
    include_heavy_object_arrays=INCLUDE_HEAVY_OBJECT_ARRAYS,
    compression="gzip",
    compression_opts=4,
)
saved_paths.append(saved_h5_path)

print("Saved database:")
for path in saved_paths:
    print(f"  [OK] {path}")
    print(f"       size={path.stat().st_size / 1024**2:.2f} MB")

Saved database:
  [OK] C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
       size=115.60 MB


In [23]:
object_db_reload, image_db_reload = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

print("Reload check:")
print("Original image_db:", len(image_db))
print("Reloaded image_db:", len(image_db_reload))
print("Original object_db:", len(object_db))
print("Reloaded object_db:", len(object_db_reload))

assert len(image_db_reload) == len(image_db)
assert len(object_db_reload) == len(object_db)

# Check one object contains required fields after reload.
sample_object_id = next(iter(object_db_reload.keys()))
sample_obj = object_db_reload[sample_object_id]

required_object_fields = [
    "object_id",
    "source_clean_key",
    "object_nut_type",
    "mask",
    "spectra",
    "mean_spectrum",
    "median_spectrum",
    "std_spectrum",
    "positions_global",
    "bbox",
    "centroid",
]

missing = [field for field in required_object_fields if field not in sample_obj]

if missing:
    raise KeyError(f"Reloaded object is missing fields: {missing}")

print("Smoke test passed.")
print("Sample object:", sample_object_id)
print("Available keys:", sorted(sample_obj.keys()))

Reload check:
Original image_db: 48
Reloaded image_db: 48
Original object_db: 1262
Reloaded object_db: 1262
Smoke test passed.
Sample object: alm1pea1_obj001
Available keys: ['area_pixels', 'batch', 'bbox', 'centroid', 'components', 'cube_crop', 'data_mode', 'description', 'image_nut_type', 'image_ref_crop', 'is_mixture', 'is_position_reference', 'is_pure', 'is_unknown', 'label_id', 'mask', 'mask_global', 'mean_spectrum', 'median_spectrum', 'n_bands', 'n_pixels', 'object_id', 'object_index', 'object_nut_type', 'position_set', 'positions_global', 'positions_local', 'sample_kind', 'source_clean_key', 'source_image', 'spectra', 'split', 'std_spectrum', 'wavelengths']


In [24]:
database_manifest_df = pd.DataFrame([{
    "raw_mat_path": str(RAW_MAT_PATH),
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "data_mode": DATA_MODE,
    "wavelength_mode": WAVELENGTH_MODE,
    "results_tag": RESULTS_TAG,

    "start_nm": float(N_START),
    "end_nm": float(N_END),
    "original_bands": int(N_BANDS_RAW),
    "n_remove_start": int(N_REMOVE_START),
    "n_stop_end": N_STOP_END,
    "n_bands_non_noisy": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),

    "min_area": int(OBJECT_MIN_AREA),
    "forced_split": FORCED_SPLIT,
    "skip_unknown": bool(SKIP_UNKNOWN),
    "include_heavy_object_arrays": bool(INCLUDE_HEAVY_OBJECT_ARRAYS),

    "n_images": int(len(image_db)),
    "n_objects": int(len(object_db)),
}])

save_parquet(database_manifest_df, DATABASE_MANIFEST_PATH)

print("Saved database manifest:")
print(DATABASE_MANIFEST_PATH)

display(database_manifest_df)

Saved database manifest:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\database_manifest.parquet


,raw_mat_path,db_h5_path,results_dir,data_mode,wavelength_mode,results_tag,start_nm,end_nm,original_bands,n_remove_start,n_stop_end,n_bands_non_noisy,min_wavelength_nm,max_wavelength_nm,min_area,forced_split,skip_unknown,include_heavy_object_arrays,n_images,n_objects
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,reflectance,non_noisy_all,non_noisy_all,889.0,1702.0,69,6,None,63,960.735294,1702.0,10,projection,True,False,48,1262


In [25]:
print("00_building_database.ipynb completed.")
print()
print("Essential outputs:")
print(" -", DB_H5_PATH)
print(" -", IMAGE_SUMMARY_PATH)
print(" -", OBJECT_SUMMARY_PATH)
print(" -", DATABASE_MANIFEST_PATH)
print()
print("Next notebook:")
print("01_database_quality_check.ipynb")

00_building_database.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\image_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\object_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\00_database\database_manifest.parquet

Next notebook:
01_database_quality_check.ipynb
